# Transcript (Text) Preprocessing

Cleans the Sinhala transcript text in `df_trim` (the audio-trimmed dataset from
`audio.ipynb`), adapting the design decisions from the team's existing 15-stage
Sinhala corpus-cleaning pipeline (`preprocess.md`) to ASR transcripts instead of
scraped web documents.

**Two things are deliberately different here vs. the source pipeline:**

1. **No document-level steps apply.** The corpus pipeline's boilerplate removal,
   document-frequency deduplication, and literal `\n`-marker repair all target
   problems specific to scraped multi-paragraph web documents. Transcripts here
   are already short, single-utterance strings pulled from structured
   audio-transcript pairs — none of that class of noise exists in this data.
2. **Code-mixed English is kept, not stripped.** The source pipeline's step 02
   deletes standalone English words because it's building a *Sinhala-only*
   tokenizer corpus. This project's audio pipeline already made the opposite
   call (`audio.ipynb`, "Remove pure-English rows" — code-mixed Sinhala/English
   rows are explicitly kept, since code-mixing is natural in Sri Lankan speech).
   So the final whitelist here allows Sinhala + Latin script, not Sinhala only.

What does carry over directly: NFC Unicode normalization first (order-independent,
establishes a consistent baseline), the Sinhala-specific character-handling
rules (combining marks, ZWJ conjunct preservation, unassigned-codepoint
rejection, the `isspace()` control-character trap), and a final whitelist +
alphabet-inspection pass as a categorical safety net rather than an
ever-growing list of one-off patches.

**Every transformation below is observed on a sample first** (raw text printed,
counts of how many rows would be affected) **before being applied**, and applied
into a new column rather than overwriting `text` in place, so the original is
never lost and each stage can be inspected independently.

In [ ]:
import os
import re
import unicodedata
from collections import Counter

import pandas as pd

# Prefer the audio-trimmed output; fall back to the raw combined dataset if audio.ipynb
# hasn't been run to completion yet, so this notebook can still be developed independently.
_CANDIDATES = ["combinedData_trimmed.parquet", "../../observation/combinedData1.parquet"]
_path = next((p for p in _CANDIDATES if os.path.exists(p)), None)
if _path is None:
    raise FileNotFoundError(f"None of {_CANDIDATES} found -- run audio.ipynb first or adjust the path.")

df_trim = pd.read_parquet(_path, columns=None)
print(f"Loaded {_path}  ->  {df_trim.shape}")
df_trim[["source_dataset", "text"]].head(5)

## Observe raw transcripts

Before touching anything: sample raw text across sources, check length distribution,
and run a full alphabet inspection (same idea as the source pipeline's step 14, but
run *first* here as a baseline instead of only at the end) to see what's actually in
the data before deciding what needs cleaning.

In [ ]:
N_SAMPLE = 5

print("=== Random sample of raw transcripts, by source ===")
for source, g in df_trim.groupby("source_dataset"):
    print(f"\n--- {source} ({len(g)} rows) ---")
    for t in g["text"].fillna("").sample(min(N_SAMPLE, len(g)), random_state=42):
        print(f"  {t!r}")

print("\n=== Transcript length (characters) ===")
print(df_trim["text"].fillna("").str.len().describe())

In [ ]:
SINHALA_START, SINHALA_END = 0x0D80, 0x0DFF
ZW_CHARS = {0x200C, 0x200D}  # ZWNJ, ZWJ -- required for Sinhala conjuncts, not disposable


def inspect_alphabet(texts, top_n=40):
    """Reusable version of the source pipeline's step 14: count every unique character
    across the given texts, with its Unicode codepoint/category/name, plus rollups by
    category so contamination (Cn unassigned, Cc control, foreign scripts) is visible
    at a glance without reading the full per-character table.
    """
    counter = Counter()
    for t in texts:
        counter.update(t)

    rows = []
    for ch, n in counter.items():
        cp = ord(ch)
        rows.append({
            "char": ch,
            "codepoint": f"U+{cp:04X}",
            "category": unicodedata.category(ch),
            "name": unicodedata.name(ch, "<unassigned>"),
            "count": n,
            "is_sinhala_block": SINHALA_START <= cp <= SINHALA_END,
        })
    report = pd.DataFrame(rows).sort_values("count", ascending=False).reset_index(drop=True)

    print(f"Unique characters: {len(report)}")
    print("\nBy category:")
    print(report.groupby("category")["count"].agg(["size", "sum"]).rename(columns={"size": "unique_chars", "sum": "total_occurrences"}))

    n_unassigned = (report["category"] == "Cn").sum()
    n_control = ((report["category"] == "Cc") & (report["char"] != "\n")).sum()
    n_space_like = (report["category"] == "Zs").sum()
    print(f"\nUnassigned (Cn) codepoints: {n_unassigned}  <- encoding corruption if > 0")
    print(f"Stray control (Cc, excl. \\n) codepoints: {n_control}  <- corruption if > 0")
    print(f"Distinct space-like (Zs) characters: {n_space_like}  <- should be 1 after whitespace cleanup")

    return report


print("=== Baseline alphabet report (raw text, before any cleaning) ===")
baseline_report = inspect_alphabet(df_trim["text"].fillna(""))
baseline_report.head(40)

## Step 1 — Unicode normalization (NFC)

Same rationale as the source pipeline's step 00: different scrapers/input methods can
represent the same visible Sinhala character sequence with different underlying
codepoint sequences (e.g. a precomposed vs. decomposed combining-mark sequence). NFC
normalization makes every occurrence canonical, so every later regex/character-class
check in this notebook can assume a single consistent representation. Runs first,
before any other step, for the same reason it does in the source pipeline: it doesn't
touch plain ASCII and has no ordering dependency on anything else.

Observe how many rows actually change under NFC before applying it.

In [ ]:
raw_text = df_trim["text"].fillna("")
nfc_preview = raw_text.apply(lambda t: unicodedata.normalize("NFC", t))

changed_mask = nfc_preview != raw_text
print(f"Rows that change under NFC normalization: {changed_mask.sum()} out of {len(raw_text)}")

for idx in df_trim[changed_mask].index[:5]:
    print(f"\n[{idx}] before: {raw_text.loc[idx]!r}")
    print(f"[{idx}] after:  {nfc_preview.loc[idx]!r}")

In [ ]:
df_trim["text_v1_nfc"] = nfc_preview
print("Applied -- df_trim['text_v1_nfc'] added. Original 'text' column left untouched.")

## Step 2 — Whitespace cleanup

Collapses repeated spaces/tabs, converts non-breaking spaces (`U+00A0`) and other
`Zs`-category space variants to a regular ASCII space, and trims leading/trailing
whitespace. Deliberately does **not** use `str.isspace()` to decide what counts as
whitespace — per the source pipeline's design notes, `isspace()` returns `True` for
some ASCII control characters (e.g. `U+001F`) that are corruption artifacts, not
real whitespace. Only actual Unicode space separators (`Zs` category) plus tab and
newline are treated as collapsible whitespace here.

Observe how many rows have non-standard whitespace before collapsing it.

In [ ]:
def is_collapsible_space(ch):
    return unicodedata.category(ch) == "Zs" or ch in ("\t", "\n", "\r")


def has_nonstandard_whitespace(t):
    return any(is_collapsible_space(ch) and ch != " " for ch in t) or t != t.strip() or "  " in t


nonstd_ws_mask = df_trim["text_v1_nfc"].apply(has_nonstandard_whitespace)
print(f"Rows with non-standard whitespace (repeated spaces, non-breaking space, "
      f"leading/trailing whitespace): {nonstd_ws_mask.sum()} out of {len(df_trim)}")

for idx in df_trim[nonstd_ws_mask].index[:5]:
    print(f"[{idx}] {df_trim.loc[idx, 'text_v1_nfc']!r}")

In [ ]:
def clean_whitespace(t):
    t = "".join(" " if (is_collapsible_space(ch) and ch != "\n") else ch for ch in t)
    t = re.sub(r"[ \t]+", " ", t)
    t = re.sub(r"\n+", "\n", t)
    return t.strip()


df_trim["text_v2_ws"] = df_trim["text_v1_nfc"].apply(clean_whitespace)

print("Applied -- df_trim['text_v2_ws'] added.")
for idx in df_trim[nonstd_ws_mask].index[:5]:
    print(f"\n[{idx}] before: {df_trim.loc[idx, 'text_v1_nfc']!r}")
    print(f"[{idx}] after:  {df_trim.loc[idx, 'text_v2_ws']!r}")

## Step 3 — Punctuation cleanup

Removes empty bracket leftovers (`()`, `[ ]`) and collapses runs of repeated or mixed
punctuation (`...`, `?!`, `--`) down to a single mark — same targeted cleanup as the
source pipeline's step 03/08. Explicitly excludes Sinhala combining marks (`Mn`/`Mc`
categories) and the zero-width joiner/non-joiner from being touched by the
punctuation regex, per the doc's warning that a naive "letters are protected,
everything else is disposable punctuation" approach silently mangles every word
with a vowel sign.

Observe how many rows are affected before applying.

In [ ]:
EMPTY_BRACKETS_RE = re.compile(r"\(\s*\)|\[\s*\]|\{\s*\}")
REPEATED_PUNCT_RE = re.compile(r"([.,!?;:\-]){2,}")


def clean_punctuation(t):
    t = EMPTY_BRACKETS_RE.sub("", t)
    t = REPEATED_PUNCT_RE.sub(r"\1", t)
    return t


punct_preview = df_trim["text_v2_ws"].apply(clean_punctuation)
punct_changed_mask = punct_preview != df_trim["text_v2_ws"]
print(f"Rows changed by punctuation cleanup: {punct_changed_mask.sum()} out of {len(df_trim)}")

for idx in df_trim[punct_changed_mask].index[:5]:
    print(f"\n[{idx}] before: {df_trim.loc[idx, 'text_v2_ws']!r}")
    print(f"[{idx}] after:  {punct_preview.loc[idx]!r}")

In [ ]:
df_trim["text_v3_punct"] = punct_preview
print("Applied -- df_trim['text_v3_punct'] added.")

## Step 4 — Emoji detection and removal

Speech transcripts are far less likely to contain emoji than web text, but crowd-sourced
or scraped-caption sources can still pick some up. Removes emoji character sequences,
including multi-codepoint emoji joined with ZWJ (e.g. family/flag emoji) — same logic
as the source pipeline's step 06: a ZWJ is only stripped when it sits directly between
two emoji codepoints, so it's never confused with the ZWJ used inside Sinhala
conjuncts (which sits between two Sinhala consonants, not emoji).

Observe how many rows actually contain emoji before deciding whether this step is
even needed for this dataset.

In [ ]:
# Common emoji Unicode blocks (pictographs, symbols, transport, flags, supplemental
# symbols, dingbats). Deliberately range-based rather than a package dependency, since
# only detection + stripping is needed here, not emoji-aware text shortening/aliasing.
EMOJI_RANGES = [
    (0x1F300, 0x1FAFF),  # misc symbols & pictographs, emoticons, transport, supplemental symbols
    (0x2600, 0x27BF),    # misc symbols, dingbats
    (0x1F1E6, 0x1F1FF),  # regional indicators (flag letters)
    (0x2190, 0x21FF),    # arrows (occasionally used decoratively, low risk here)
    (0xFE0F, 0xFE0F),    # variation selector-16 (emoji presentation)
]


def is_emoji(ch):
    cp = ord(ch)
    return any(lo <= cp <= hi for lo, hi in EMOJI_RANGES)


def has_emoji(t):
    return any(is_emoji(ch) for ch in t)


emoji_mask = df_trim["text_v3_punct"].apply(has_emoji)
print(f"Rows containing emoji: {emoji_mask.sum()} out of {len(df_trim)}")

for idx in df_trim[emoji_mask].index[:5]:
    print(f"[{idx}] {df_trim.loc[idx, 'text_v3_punct']!r}")

In [ ]:
def remove_emoji(t):
    out = []
    for i, ch in enumerate(t):
        if is_emoji(ch):
            continue
        # Drop a ZWJ only when it sits directly between two emoji codepoints -- leaves
        # Sinhala-conjunct ZWJ (between two Sinhala consonants) completely untouched.
        if ord(ch) == 0x200D and i > 0 and i + 1 < len(t) and is_emoji(t[i - 1]) and is_emoji(t[i + 1]):
            continue
        out.append(ch)
    return clean_whitespace("".join(out))


df_trim["text_v4_emoji"] = df_trim["text_v3_punct"].apply(remove_emoji)

print("Applied -- df_trim['text_v4_emoji'] added.")
for idx in df_trim[emoji_mask].index[:5]:
    print(f"\n[{idx}] before: {df_trim.loc[idx, 'text_v3_punct']!r}")
    print(f"[{idx}] after:  {df_trim.loc[idx, 'text_v4_emoji']!r}")

## Step 5 — Final character whitelist (safety net)

Same rationale as the source pipeline's step 13: the targeted steps above only
guarantee that the specific patterns they were built for are gone, not what
character set survives overall. This final pass is a categorical safety net against
anything unanticipated — leftover Tamil/Arabic/etc. script fragments,
private-use-area and unassigned codepoints (encoding corruption), stray symbols.

**Whitelist for this project (Sinhala + Latin, not Sinhala-only):**
- Sinhala Unicode block, assigned codepoints only (rejects `Cn` unassigned gaps like
  `U+0DB2`, per the doc's encoding-corruption warning)
- ZWJ / ZWNJ (`U+200D` / `U+200C`) — required for Sinhala conjuncts
- ASCII Latin letters (`A-Za-z`) and digits — kept, unlike the source pipeline,
  because code-mixed English is intentionally retained in this dataset
- A small fixed punctuation set: `. , ! ? ; : ' " ( ) - /`
- Normalized whitespace (single space, single newline)

Same visibility-based removal rule as the source doc: invisible/control characters
(`Cf`, `Cc`, `Cn`) are deleted outright (no replacement), since inserting a visible
space where an invisible character used to be would introduce a word break that
never existed. Visible disallowed characters (foreign scripts, stray symbols) are
replaced with a single space, so two words separated only by the removed character
don't fuse together.

Observe what this would strip (dry run) before applying, since it's the most
aggressive step in the pipeline.

In [ ]:
ALLOWED_PUNCTUATION = set(".,!?;:'\"()-/")


def is_allowed_char(ch):
    cp = ord(ch)
    if SINHALA_START <= cp <= SINHALA_END:
        return unicodedata.category(ch) != "Cn"  # reject unassigned gaps in the block
    if cp in ZW_CHARS:
        return True
    if ch.isascii() and (ch.isalpha() or ch.isdigit()):
        return True
    if ch in ALLOWED_PUNCTUATION:
        return True
    if ch in (" ", "\n"):
        return True
    return False


def whitelist_dry_run(t):
    """Returns (would_change, disallowed_chars_found) without modifying anything."""
    disallowed = {ch for ch in t if not is_allowed_char(ch)}
    return bool(disallowed), disallowed


dry_run = df_trim["text_v4_emoji"].apply(whitelist_dry_run)
would_change_mask = dry_run.apply(lambda r: r[0])
all_disallowed = Counter()
for _, disallowed in dry_run:
    all_disallowed.update(disallowed)

print(f"Rows the whitelist would change: {would_change_mask.sum()} out of {len(df_trim)}")
print(f"\nDisallowed characters found (would be stripped), by frequency:")
for ch, n in all_disallowed.most_common(30):
    cp = ord(ch)
    print(f"  {ch!r}  U+{cp:04X}  {unicodedata.category(ch)}  {unicodedata.name(ch, '<unassigned>')}  x{n}")

for idx in df_trim[would_change_mask].index[:5]:
    print(f"\n[{idx}] {df_trim.loc[idx, 'text_v4_emoji']!r}")

In [ ]:
INVISIBLE_CATEGORIES = {"Cf", "Cc", "Cn"}


def apply_whitelist(t):
    out = []
    for ch in t:
        if is_allowed_char(ch):
            out.append(ch)
        elif unicodedata.category(ch) in INVISIBLE_CATEGORIES:
            pass  # delete outright -- no replacement, avoids inserting a false word break
        else:
            out.append(" ")  # visible disallowed char -- replace with space, avoids word fusion
    return clean_whitespace("".join(out))


df_trim["text_v5_whitelist"] = df_trim["text_v4_emoji"].apply(apply_whitelist)

print("Applied -- df_trim['text_v5_whitelist'] added.")
for idx in df_trim[would_change_mask].index[:5]:
    print(f"\n[{idx}] before: {df_trim.loc[idx, 'text_v4_emoji']!r}")
    print(f"[{idx}] after:  {df_trim.loc[idx, 'text_v5_whitelist']!r}")

## Verify the output (final alphabet report)

Same check as the source pipeline's step 14, run again here on the cleaned text as
a final QA gate. Compare against the baseline report from the top of this notebook:
unique character count should have dropped sharply, `Cn`/stray `Cc` entries should
be gone, and exactly one space-like (`Zs`) character should remain.

In [ ]:
print(f"=== Final alphabet report (cleaned text) ===")
final_report = inspect_alphabet(df_trim["text_v5_whitelist"])

print(f"\nUnique chars: {len(baseline_report)} (baseline) -> {len(final_report)} (cleaned)")

emptied_mask = (df_trim["text_v5_whitelist"].str.strip() == "") & (df_trim["text"].fillna("").str.strip() != "")
print(f"\nRows that became empty after cleaning (whole utterance was junk): {emptied_mask.sum()}")
for idx in df_trim[emptied_mask].index[:10]:
    print(f"  [{idx}] original: {df_trim.loc[idx, 'text']!r}")

final_report.head(40)

## Commit cleaned text + drop emptied rows (NOT APPLIED)

**Status: observation only.** `df_trim` still has the original `text` column plus all
five intermediate `text_v1`...`text_v5_whitelist` columns — nothing is overwritten or
dropped. Review the emptied-row samples above first; uncomment below once satisfied.

In [ ]:
# df_trim = df_trim[~emptied_mask].reset_index(drop=True)
# df_trim["text"] = df_trim["text_v5_whitelist"]
# df_trim = df_trim.drop(columns=["text_v1_nfc", "text_v2_ws", "text_v3_punct", "text_v4_emoji", "text_v5_whitelist"])
# print(f"New shape: {df_trim.shape}")
#
# out_path = "combinedData_transcript_cleaned.parquet"
# df_trim.to_parquet(out_path, index=False)
# print(f"Saved cleaned dataset to {out_path}")

print(f"Would drop {emptied_mask.sum()} emptied rows and commit text_v5_whitelist -> text (not applied)")